# lenia-vlm on Colab: MaCE-Lenia runs

GPU runs for the live pipeline (see README.md). Results go to your Drive folder `lenia-runs/` and survive
disconnects; every run resumes if you re-run sections 1-2 and then the same command with the same `--out`.
The earlier 19-section notebook (classic Lenia, distillation, hill climber, MAP-Elites v1-v4) is in
`archive/colab/lenia_colab.ipynb`.

## 1. GPU and Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import os
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lenia-runs'      # every result goes here and survives disconnects
os.makedirs(RUNS, exist_ok=True)
print('results folder:', RUNS)

## 2. Code, dependencies, judge keys

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO, DIR = 'github.com/feli10/lenia-vlm.git', '/content/lenia-vlm'
tok = secret('GITHUB_TOKEN')
url = f'https://x-access-token:{tok}@{REPO}' if tok else f'https://{REPO}'
if not os.path.isdir(DIR):
    r = subprocess.run(['git', 'clone', '-q', url, DIR], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', DIR, 'pull', '-q', url, 'main'], capture_output=True, text=True)
print('git:', 'ok' if r.returncode == 0 else r.stderr.replace(tok or '\0', '***'))
subprocess.run(['git', '-C', DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'])   # don't keep the token in .git
os.chdir(DIR)
!git log --oneline -1
!pip install -q -e .

In [ ]:
# API keys for the judges, if you added them as Colab secrets
keys = {k: secret(k) for k in ('OPENROUTER_API_KEY', 'TYPESAFE_API_KEY', 'GOOGLE_CLOUD_PROJECT')}
with open('.env', 'w') as f:
    for k, v in keys.items():
        if v:
            f.write(f'{k}={v}\n')
print('judge keys present:', [k for k, v in keys.items() if v] or 'none (fine for simulation-only runs)')

## 3. Checks

The test suite, offline: the golden test (prompts, pictures, cache keys, mutation and simulation unchanged, so
old caches and runs stay valid) and the PyTorch-vs-numpy parity tests. Do not use the GPU for runs unless these
pass.

In [ ]:
!pip install -q pytest && python -m pytest -q tests

## 4. Base search: MAP-Elites + surprise archive

The command that produced `mace-qd5-sarch-s0`, the seed 0 base run the stage-1 core test reuses (4 islands,
bodies x form x motion, complexity entry duel, surprise archive). About 2.3 h and ~$3 for 100 iterations.
Kept for reference; the seed 1 base run for stage 1 (shorter, with the record logging) is added here when it
is built.

In [ ]:
# reference: seed 0 (already done)
# !cd /content/lenia-vlm && python scripts/mace_qd.py run --backend auto --islands 4 --axes bodies,form,motion --body-edges 3,9 --motion-edges 0.75,1.0,1.25 --iterations 100 --sarch --seed 0 --out {RUNS}/mace-qd5-sarch-s0

### 4b. Map-prompt test (Jev ~$2.90, ~2.3 h on a T4; 11-04)
Seed 0's base run again, with one change: the entry duel's goal sentence is the user's aim instead of "complex, life-like" (`--aim`, `lenia.prompts.map_prompt`). `--like` copies every other setting and the form-axis fit from the base run, so the two maps' cells mean the same thing. The base run is the other arm (not re-run). Resumes after a disconnect. `--sim-batch` is free to raise on a bigger GPU (lower it if it runs out of memory).

In [ ]:
BASE = f'{RUNS}/mace-qd5-sarch-s0'
AIM = "more interesting as artificial life: richer structure or behaviour, more like something alive"
!cd /content/lenia-vlm && python scripts/mace_qd.py run --out {BASE}-aim --like {BASE} --backend auto --aim "{AIM}"
!cd /content/lenia-vlm && python scripts/maptest.py stats --runs {BASE},{BASE}-aim --dest {BASE}-aim/maptest

In [ ]:
# the blind session (GPU, no API calls): 30 interest pairs, the two maps' elites in the same island and cell.
# Then download mace-qd5-sarch-s0-aim/maptest/ from Drive to the Mac's runs/mace-qd5-sarch-s0-aim/maptest and run
# `maptest.py rate --aim runs/mace-qd5-sarch-s0-aim`, then `maptest.py report --aim runs/mace-qd5-sarch-s0-aim`.
BASE = f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/maptest.py build --base {BASE} --aim {BASE}-aim --backend auto

## 5. Stage 1: expeditions (seed 0)

Each arm runs one 10-step expedition (32 gentle mutants a step, judged against the trigger's start, comma selection: expedition v3) from each of the 12 triggers in `stage1-s0/triggers.json`.
Copy the Mac's `runs/stage1-s0` folder to Drive `lenia-runs/stage1-s0` first (triggers, pictures, the shared judge and
VLM caches). Run the arms in order: **L** (then `check`: gate 0c, the floor), then **N**, **NR**, **NS**. NR and NS
reuse N's first 5 steps from the caches and ask Gemini Pro for one review per trigger, which needs the Gemini cell
below (Vertex AI, billed to your Google Cloud project; add `GOOGLE_CLOUD_PROJECT` as a Colab secret). Every command
resumes where it stopped.

Results go to `stage1-s0/exp-v3/<arm>/` (v2 in `exp-v2/`, v1 in `arms/`, kept as the record). The simulation cache lives on local disk (`--evals /content/evals`); accepted worlds are copied to Drive.

In [ ]:
# Gemini on Vertex (arms NR and NS; arm N's names are already cached): sign in once per session.
# No google-genai install: lenia.describe falls back to Vertex's REST API with Colab's own google-auth
# (installing the SDK upgrades the google-auth that Colab pins).
# The Google account signed in here must have the role Vertex AI User on the project in GOOGLE_CLOUD_PROJECT
# (11-04: the project belongs to a different account than Colab/Drive; a 403 PERMISSION_DENIED means it lacks that role).
from google.colab import auth
auth.authenticate_user()

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py expedition --arm L --run {BASE} --out {S1} --backend auto --evals /content/evals

In [ ]:
# held-out Luna, end vs start (gate 0c for arm L); no GPU needed, can also run on the Mac
!cd /content/lenia-vlm && python scripts/stage1.py check --arm L --run {BASE} --out {S1}

In [ ]:
# after gate 0c passes and Felix has looked at exp-v3/L/check.png:
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm N  --run {BASE} --out {S1} --backend auto --evals /content/evals
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm NR --run {BASE} --out {S1} --backend auto --evals /content/evals
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm NS --run {BASE} --out {S1} --backend auto --evals /content/evals
# !cd /content/lenia-vlm && for A in N NR NS; do python scripts/stage1.py check --arm $A --run {BASE} --out {S1}; done

### 5b. Caption check (v4 step 2; no API calls)
Which CLIP score of a child against its goal caption agrees with Jev: the target (cos to the end caption) or the change (directional CLIP, start -> end)? Regrows expedition v3's arm L children for the 6 captioned goals (about 1,900 worlds), checks they reproduce (keys and pictures of the accepted children), and writes `stage1-s0/capcheck/report.md`. The regrown pictures stay on local disk; nothing is added to `stage1-s0/evals`.

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py capcheck --run {BASE} --out {S1} --backend auto --evals /content/capcheck-evals

### 5c. Expedition v4 (Jev ~$2.15 in all; run every arm in this one session)
48 children a step (mutation 2.0 -> 0.8 over 20 steps), a shortlist of 16 (CLIP by goal caption, or the motion floor plus a random 16 when the caption is "none"), Jev picks against the start with the complexity floor. Arms: L (library goals), N (VLM goals), NF (N's goals, floor route, on L's 6 uncaptioned triggers), C (CLIP only, no Jev). Results in `stage1-s0/exp-v4/<arm>/` (`sheet.png`: start | halfway | end; `summary.md`). Resumes after a disconnect.

First the smoke test (2 triggers, 3 steps, about $0.02): check it runs and that the sheet looks sane, then the arms.

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py v4 --arm N --run {BASE} --out {S1} --backend auto --evals /content/evals --tag exp-v4-smoke --limit 2 --stop-after 3
!cat {S1}/exp-v4-smoke/N/summary.md
from IPython.display import Image as _I, display
display(_I(f'{S1}/exp-v4-smoke/N/sheet.png'))

In [ ]:
# a Python loop: in a `!` line, a shell variable ($A) or any undefined name stops Colab from filling in {BASE} and {S1}
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'   # again here: a new runtime skips the smoke cell
for A in ['N', 'L', 'NF', 'C']:
    !cd /content/lenia-vlm && python scripts/stage1.py v4 --arm {A} --run {BASE} --out {S1} --backend auto --evals /content/evals
    if _exit_code != 0:
        break
for A in ['N', 'L', 'NF', 'C']:
    !cat {S1}/exp-v4/{A}/summary.md

### 5d. Blind rating session for v4 (GPU only, no API calls)
Picks the pairs, regrows every world in them as a film, and writes `stage1-s0/rate-v4/` (tasks, films, stills, replay check). Then copy that one folder from Drive to the Mac's `runs/stage1-s0/rate-v4` and rate there (`stage1.py v4rate rate --out runs/stage1-s0`).

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py v4rate build --out {S1} --run {BASE} --backend auto

### 5e. Arm N-aim (Jev ~$0.45; 11-04)
Aim-conditioned goals (`namer_aim_v3`) on the same 12 triggers, 10 steps of the 20-step mutation schedule, so its mutations are exactly L's and N's first 10. Needs two Mac files uploaded to Drive `lenia-runs/stage1-s0/` first: `goals-aim/N-aim.json` and `captions/N-aim.json` (plus `captions/N-aim.md`). Resumes after a disconnect.

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py v4 --arm N-aim --run {BASE} --out {S1} --backend auto --evals /content/evals
!cat {S1}/exp-v4/N-aim/summary.md
from IPython.display import Image as _I, display
display(_I(f'{S1}/exp-v4/N-aim/sheet.png'))

### 5f. Gemini steps on Colab (aim namer, captions; 11-04)
Goals and captions are written straight into Drive's `stage1-s0`, so nothing moves Mac → Drive. Run the Gemini sign-in cell at the top of section 5 first (and `GOOGLE_CLOUD_PROJECT` must be a Colab secret).

**Once, before the first Gemini step here:** the Mac's Gemini caches hold answers Drive's lack (the aim namer v1–v3, N-aim's captions). Upload the Mac's `runs/stage1-s0/describe_cache.sqlite` to Drive as `stage1-s0/describe_cache_mac.sqlite` and `runs/stage1-s0/captions/caption_cache.sqlite` as `stage1-s0/captions/caption_cache_mac.sqlite` (new names, so nothing on Drive is replaced), then run the merge cell. It adds only missing answers and backs up Drive's caches first. From then on the Mac's copies are not used for Gemini steps.

In [ ]:
S1 = f'{RUNS}/stage1-s0'
!cd /content/lenia-vlm && python scripts/stage1.py mergecache --src {S1}/describe_cache_mac.sqlite --dst {S1}/describe_cache.sqlite
!cd /content/lenia-vlm && python scripts/stage1.py mergecache --src {S1}/captions/caption_cache_mac.sqlite --dst {S1}/captions/caption_cache.sqlite

In [ ]:
# aim-conditioned goals (lenia.prompts.NAMER_AIM), beside v1's and the library's; cached answers cost nothing.
# A new namer version overwrites goals-aim/: rename the old folder first, and pass --prev {S1}/goals-aim-vN/N-aim.json
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py aimgoals --run {BASE} --out {S1}
from IPython.display import Image as _I, display
display(_I(f'{S1}/goals-aim/compare.png'))

In [ ]:
# captions for the aim goals (they decide each trigger's route in arm N-aim)
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py captions --arm N-aim --run {BASE} --out {S1}

### 5g. Blind rating session for N-aim (GPU only, no API calls)
48 pairs in `stage1-s0/rate-aim/`: interest N-aim vs L and vs N at step 10 (12 each), N-aim end vs start and control vs start on N-aim's goal (12 each). Then download that one folder from Drive to the Mac's `runs/stage1-s0/rate-aim` and rate there: `stage1.py v4rate rate --session aim --out runs/stage1-s0`; `v4rate report --session aim` splits interest by goal type.

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py v4rate build --session aim --out {S1} --run {BASE} --backend auto

### 5h. Arm NS-aim: N-aim plus one aim-aware review (Gemini ~$0.20, Jev up to ~$0.25; 11-04)
Run the Gemini sign-in cell at the top of section 5 first. NS-aim starts from N-aim's state at step 5 (copied, not regrown), asks Gemini to review each trigger (`review_aim_v1`: the start, the current parent and the 3 most surprising unchosen children of steps 3-5; continue, refine or switch to one of them), captions any new goal, then runs steps 6-10. Triggers that continue copy N-aim's steps 6-10 (they would replay it exactly), so only changed triggers are simulated. Decisions in `exp-v4/NS-aim/review.md`. Resumes after a disconnect.

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py v4 --arm NS-aim --run {BASE} --out {S1} --backend auto --evals /content/evals
!cat {S1}/exp-v4/NS-aim/review.md {S1}/exp-v4/NS-aim/summary.md
from IPython.display import Image as _I, display
display(_I(f'{S1}/exp-v4/NS-aim/sheet.png'))

### 5i. Blind rating session for NS-aim (GPU only, no API calls)
Only the triggers whose review changed something: interest NS-aim vs N-aim at step 10, and NS-aim's end vs the world its new goal was judged against (the start for refine, the new stepping stone for switch). Writes `stage1-s0/rate-ns/`; download it to the Mac's `runs/stage1-s0/rate-ns` and rate there (`stage1.py v4rate rate --session ns --out runs/stage1-s0`).

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py v4rate build --session ns --out {S1} --run {BASE} --backend auto

## 6. CLIP targeted-search bench (no API calls)

Which search settings actually steer a world toward a text target? CLIP scores each child (final snapshot vs the
prompt), so it costs nothing but GPU time (~1-1.5 h on a T4 for the default 6 configurations x 4 targets x 2 starts
x 12 steps x 16 children). Compares world inheritance vs regrowth from noise, mutation strength, sep-CMA-ES (ASAL's
optimiser) and a best-of-N local control. Resumes after a disconnect. See `scripts/clipbench.py`.

In [ ]:
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the base run (start worlds); defined here so this section runs on its own
!pip install -q cma
!cd /content/lenia-vlm && python scripts/clipbench.py run --run {BASE} --out {RUNS}/clipbench --backend auto

In [ ]:
# Round 1 before/after films (one GIF per target; rows = start worlds, columns = start | best per setting). A few min on GPU.
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the start worlds (defined in every cell, so each runs on its own)
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {RUNS}/clipbench --backend auto
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{RUNS}/clipbench/gifs/*.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Round 2: can Jev steer? Jev selects (goal score vs the start), CLIP scores every child, held out. ~$0.60 of Jev, ~1 h (20 steps: CLIP's climb plateaus ~22).
# Needs .env with the TypeSafe key in /content/lenia-vlm (as for the expeditions). Resumable: re-run the cell if it stops.
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the start worlds (defined in every cell, so each runs on its own)
!cd /content/lenia-vlm && LENIA_VLM_CACHE_ONLY=1 python scripts/clipbench.py run --run {BASE} --out {RUNS}/clipbench-jev --backend auto --configs es-regrow-1,jev-regrow-1,local-1 --steps 20
!cd /content/lenia-vlm && python scripts/clipbench.py report --out {RUNS}/clipbench-jev

In [ ]:
# Round 2 films: start | CLIP-steered best | Jev-steered best, per target (rows = starts). Also re-prints the report
# with Jev's own scores per step. A few min on GPU, no API calls.
BASE = f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/clipbench.py report --out {RUNS}/clipbench-jev | grep -A3 'Jev-steered'
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {RUNS}/clipbench-jev --backend auto --configs es-regrow-1,jev-regrow-1
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{RUNS}/clipbench-jev/gifs/*.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Long run: does the climb keep going past 12 steps, and can it steer far (pizza)? CLIP only, free, ~35 min. Resumable.
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the start worlds (defined in every cell, so each runs on its own)
!cd /content/lenia-vlm && python scripts/clipbench.py run --run {BASE} --out {RUNS}/clipbench-long --backend auto \
    --configs es-regrow-1 --steps 40 --prompts "a worm|a ring|a starfish|a pepperoni pizza"
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {RUNS}/clipbench-long --backend auto --configs es-regrow-1
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{RUNS}/clipbench-long/gifs/*.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Is CLIP's steering target-specific, or just making worlds more CLIP-friendly in general? ~1 min, free.
!cd /content/lenia-vlm && python scripts/clipbench.py cross --out {RUNS}/clipbench-long
!cd /content/lenia-vlm && python scripts/clipbench.py cross --out {RUNS}/clipbench | tail -8

In [ ]:
# Round 3: sharpen Jev's choice and fight the plateau. Each config differs from jev-regrow-1 (round 2's Jev setting) in one
# thing: playoff (top 4 meet head-to-head), clipfilter (48 children, CLIP shortlists 16, Jev picks), sched (mutation
# 2.0 -> 0.5). Same targets, starts and mutants as round 2. jev-regrow-1 reuses round 2's cached Jev answers.
# GIFs are BLIND: columns lettered and shuffled per target, key in gifs/key.json (rate before opening it).
# ~$1.9 of new Jev (cap $3.5 includes the copied round-2 answers); ~2.5-3 h on a T4. Resumable: re-run the cell.
import os, shutil
BASE = f'{RUNS}/mace-qd5-sarch-s0'
R3 = f'{RUNS}/clipbench-r3'
os.makedirs(R3, exist_ok=True)
if os.path.exists(f'{RUNS}/clipbench-jev/judge_cache.sqlite') and not os.path.exists(f'{R3}/judge_cache.sqlite'):
    shutil.copy(f'{RUNS}/clipbench-jev/judge_cache.sqlite', f'{R3}/judge_cache.sqlite')
!cd /content/lenia-vlm && LENIA_VLM_CACHE_ONLY=1 python scripts/clipbench.py run --run {BASE} --out {R3} --backend auto \
    --configs jev-regrow-1,jev-playoff-1,jev-clipfilter-1,jev-sched --steps 20 --max-usd 3.5
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {R3} --backend auto --blind \
    --configs jev-regrow-1,jev-playoff-1,jev-clipfilter-1,jev-sched
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{R3}/gifs/*_blind.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Head-to-head, 40 steps: the candidate expedition step (jev-v4: CLIP shortlist + motion floor, Jev chooses, mutation
# 2.0 -> 0.8) vs the original CLIP-driven search (es-regrow-1), same mutant seeds, on worm / ring / starfish / pizza.
# ~$1.15 of Jev (cap $2); ~3.5-4 h on a T4 (jev-v4 grows 48 children a step). Resumable: re-run the cell.
# GIFs are blind (key in gifs/key.json); afterwards rate on the Mac with `clipbench.py rate --out runs/clipbench-v4`.
BASE = f'{RUNS}/mace-qd5-sarch-s0'
V4 = f'{RUNS}/clipbench-v4'
!cd /content/lenia-vlm && LENIA_VLM_CACHE_ONLY=1 python scripts/clipbench.py run --run {BASE} --out {V4} --backend auto \
    --configs es-regrow-1,jev-v4 --steps 40 --prompts "a worm|a ring|a starfish|a pepperoni pizza" --max-usd 2
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {V4} --backend auto --blind --configs es-regrow-1,jev-v4
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{V4}/gifs/*_blind.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Before/after films of the Jev-driven 40-step run only (start | jev-v4's best world), labelled, not blind. ~5 min.
BASE = f'{RUNS}/mace-qd5-sarch-s0'
V4 = f'{RUNS}/clipbench-v4'
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {V4} --backend auto --configs jev-v4
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{V4}/gifs/*.gif')):
    if not f.endswith('_blind.gif'):
        print(f); display(GIF(filename=f))

In [ ]:
# Round 4: why did the CLIP shortlist help, and does re-anchoring fix Jev's saturated ruler? Round 3's targets, starts
# and mutants, 20 steps. jev-regrow-1 and jev-clipfilter-1 reuse round 3's cached answers; new: jev-floor (motion
# floor, random 16 of 48, no CLIP) and jev-milestone (reference moves to the parent at >= 0.85, at most every 5 steps).
# ~$1.10 of new Jev (the cap counts the copied round-3 answers too); ~2.5 h on a T4. Resumable: re-run the cell.
# GIFs are blind (key in gifs/key.json). Then rate on the Mac: clipbench.py rate --out runs/clipbench-r4
import os, shutil
BASE = f'{RUNS}/mace-qd5-sarch-s0'
R4 = f'{RUNS}/clipbench-r4'
os.makedirs(R4, exist_ok=True)
if os.path.exists(f'{RUNS}/clipbench-r3/judge_cache.sqlite') and not os.path.exists(f'{R4}/judge_cache.sqlite'):
    shutil.copy(f'{RUNS}/clipbench-r3/judge_cache.sqlite', f'{R4}/judge_cache.sqlite')
!cd /content/lenia-vlm && LENIA_VLM_CACHE_ONLY=1 python scripts/clipbench.py run --run {BASE} --out {R4} --backend auto \
    --configs jev-regrow-1,jev-clipfilter-1,jev-floor,jev-milestone --steps 20 --max-usd 4.5
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {R4} --backend auto --blind \
    --configs jev-regrow-1,jev-clipfilter-1,jev-floor,jev-milestone
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{R4}/gifs/*_blind.gif')):
    print(f); display(GIF(filename=f))

## 7. Full run, seed 1 (11-05; `scripts/fullrun.py`)

One aim, end to end, in 5 rounds: 20 map iterations (seed 1, the aim's entry duel, surprise duel at 484 px), 10 triggers from map elites + surprise archive (re-scored against current elites and the 2 nearest explored worlds), aim-conditioned goals and captions (Gemini), 10 NS-aim expeditions with one review, and feedback of each expedition's end into the map. Then the gallery (Bradley-Terry from the aim entry duels, a Swiss tournament among cell champions, a diversified shortlist of 24, Gemini picks 8) and `report.md`. Everything resumes after a disconnect: re-run sections 1-2, the Gemini sign-in, then the same command.

Expected: Jev about $4.80 for the full run and $1.90 for the baseline; Gemini about $1.70 (credit); GPU about 8 h + 1.8 h on a T4 (raise `--sim-batch` on a bigger GPU).

In [ ]:
# Gemini on Vertex (goals, captions, reviews, the gallery pick): sign in once per session (account with Agent Platform User)
from google.colab import auth
auth.authenticate_user()

### 7a. Smoke test of round 1 first (Jev about $0.30, Gemini about $0.10, about 20 min)
A separate folder: 3 map iterations, 2 triggers, 6 expedition steps (the review at step 5 still runs), feedback, gallery and report. Read `report.md`, `rounds/r1/goals.md` and `rounds/r1/exp/review.md`, then move the folder aside.

In [ ]:
SMOKE = f'{RUNS}/full-s1-smoke'
!cd /content/lenia-vlm && python scripts/fullrun.py run --out {SMOKE} --backend auto --evals /content/evals-smoke --rounds 1 --iters 2 --triggers 2 --end 6
!cat {SMOKE}/rounds/r1/goals.md {SMOKE}/rounds/r1/exp/review.md {SMOKE}/report.md
from IPython.display import Image as _I, display
display(_I(f'{SMOKE}/rounds/r1/triggers.png')); display(_I(f'{SMOKE}/gallery/gallery.png'))

### 7b. The full run (resumable; `--baseline` forks the map after round 1's map search)

In [ ]:
FULL, BASE1 = f'{RUNS}/full-s1', f'{RUNS}/base-s1'
!cd /content/lenia-vlm && python scripts/fullrun.py run --out {FULL} --backend auto --evals /content/evals --baseline {BASE1}
!cat {FULL}/report.md
from IPython.display import Image as _I, display
display(_I(f'{FULL}/gallery/gallery.png'))

### 7c. The baseline: the forked map continues alone to iteration 100, then its own gallery (needs the Gemini sign-in)
Can run in another Colab session while 7b runs (different folders). Then the report with both, and the ledger.

In [ ]:
FULL, BASE1 = f'{RUNS}/full-s1', f'{RUNS}/base-s1'
!cd /content/lenia-vlm && python scripts/mace_qd.py run --out {BASE1} --backend auto --iterations 100
!cd /content/lenia-vlm && python scripts/fullrun.py gallery --run {BASE1} --out {BASE1}
!cd /content/lenia-vlm && python scripts/fullrun.py report --out {FULL} --baseline {BASE1}
!cd /content/lenia-vlm && python scripts/fullrun.py ledger --out {FULL}
from IPython.display import Image as _I, display
display(_I(f'{BASE1}/gallery/gallery.png'))

### 7d. Check the full run from what it stored (Jev about $0.12, no Gemini; works on an unfinished run)
`scripts/fullcheck.py`: per-round table; every finished expedition's end vs its own start on the aim duel, with controls; the ends on neutral ground (a Swiss tournament with the map's cell champions); the map, start | end stills and GIFs of the clearest wins and losses and the top-rated worlds. Output in `full-s1/check/`.

In [ ]:
FULL = f'{RUNS}/full-s1'
!cd /content/lenia-vlm && python scripts/fullcheck.py --out {FULL} --backend auto --evals /content/evals
import glob
from IPython.display import Image as _I, display
display(_I(f'{FULL}/check/map.png')); display(_I(f'{FULL}/check/pairs.png'))
for g in sorted(glob.glob(f'{FULL}/check/gif/*.gif')):
    print(g.split('/')[-1]); display(_I(filename=g, format='gif'))

### 7e. Blind rating session + judge test (GPU only, no API calls here; 11-06)
`scripts/fullrate.py build` picks the pairs (goal steering: end vs reference with controls; refinement: end vs start, end vs step 5, map replacements; calibration: cross-cell elites, gallery vs random elites) and grows every world's film into `full-s1/rate/`. Then on the Mac: `rate` (the page), and only after rating, `judges` (Jev and Gemini on the same pairs) and `report`.

In [ ]:
FULL = f'{RUNS}/full-s1'
!cd /content/lenia-vlm && python scripts/fullrate.py build --out {FULL} --backend auto
!cat {FULL}/rate/design.md

## 8. Breeder: evolve worlds by hand, Picbreeder style (GPU only, no API calls; 11-06)
`scripts/breed.py serve` grows each generation (the parent's mutants) on the GPU and serves the page in a new browser tab. Click a child to breed from it, click the parent for new children, pick the mutation strength (0.3 small … 3 = the map's), star what you like. Starts from random founders or seed 1's map elites. The session and every pick are saved in `full-s1/breed/` and resume after a disconnect (re-run sections 1–2, then this cell). On the Mac the same command runs on the Apple GPU (see the script's docstring). Stop it with `breeder.terminate()`.

In [ ]:
import subprocess, sys, time
from google.colab import output
FULL = f'{RUNS}/full-s1'
breeder = subprocess.Popen([sys.executable, 'scripts/breed.py', 'serve', '--run', FULL, '--backend', 'auto', '--no-browser', '--port', '8776'],
                           cwd='/content/lenia-vlm', stdout=open('/content/breed.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10); print(open('/content/breed.log').read())
output.serve_kernel_port_as_window(8776)

## 9. Motion benchmark, Stage A: can Jev see motion? (GPU only, no API calls; 11-07)
`scripts/motionbench.py build` regrows about 550 seed-1 worlds (final map elites, map entry duels, expedition siblings), measures how much each changes between its four snapshots, picks the pairs (easy and close change pairs, frozen copies; dev and test) and grows 10 films for your truth check. Roughly 20 min on a T4, about 150 MB in `motion-bench/`. Then on the Mac: `check`, `check-report`, `judge --split dev`, `report` (the script's docstring).


In [ ]:
FULL, MB = f'{RUNS}/full-s1', f'{RUNS}/motion-bench'
!cd /content/lenia-vlm && python scripts/motionbench.py build --source {FULL} --out {MB} --backend auto
!cat {MB}/pool.md

## 10. Rating benchmark: can Jev make Felix's calls from stills? (GPU only, no API calls; 11-10)
`scripts/ratebench.py build` regrows the 113 worlds Felix rated in the seed-1 session (their rules are recovered by replaying the session's pair selection) and stores their frames in `rate-bench/`. It must run here: the session's worlds were grown on this GPU, and a CPU regrowth differs. It stops if any world doesn't match the stills Felix was shown. A few minutes. Then on the Mac: `judge` and `report` (the script's docstring).


In [ ]:
FULL, RB = f'{RUNS}/full-s1', f'{RUNS}/rate-bench'
!cd /content/lenia-vlm && python scripts/ratebench.py build --source {FULL} --out {RB} --backend auto

## Notes

- The free tier disconnects idle sessions and caps sessions at about 12 hours. After a disconnect the machine
  is wiped, your Drive is not: re-run sections 1-2, then the same command with the same `--out`.
- Judge calls made here are cached in the run folder (`judge_cache.sqlite`), so a resumed run does not pay twice.
- Claude can read your Drive folder: say when a run has finished.